# 02 — Feature Engineering Demo

Walks through the eleven leak-free feature groups implemented in `src/features/`, orchestrated by `FeatureEngineer` (`src/features/feature_engineer.py`). Full formulas and rationale: `docs/feature_engineering.md`.

In [ ]:
from src.utils.io import read_table, write_table
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.features.feature_engineer import FeatureEngineer
from src.utils.config import load_training_config, resolve_path

cfg = load_training_config()
df = read_table(resolve_path("processed_dataset"))
df.shape

## Run the full FeatureEngineer

`FeatureEngineer` is a scikit-learn `BaseEstimator`/`TransformerMixin`, so it can be used standalone (as here) or dropped straight into a `Pipeline` (see `src/models/pipeline.py::get_feature_engineered_pipeline`, used by `predict.py`).

In [ ]:
fe = FeatureEngineer(top_n_countries=cfg["top_n_countries"])
engineered = fe.fit_transform(df)
new_columns = [c for c in engineered.columns if c not in df.columns]
print(f"{len(new_columns)} engineered columns:")
new_columns

## Inspect the most important engineered feature: `revpar_proxy`

`revpar_proxy = ttm_occupancy * ttm_avg_rate` mirrors the hotel-industry RevPAR KPI, and is consistently the single most important predictor for both tree-based models (§4.3 of the technical report).

In [ ]:
engineered[["ttm_occupancy", "ttm_avg_rate", "revpar_proxy", "ttm_revenue"]].describe()

## Sanity check: no feature reads the target

This is the same invariant `tests/test_features.py` asserts — no column name (besides the target itself) references `revenue`.

In [ ]:
leaky = [c for c in engineered.columns if "revenue" in c.lower() and c != cfg["target"]]
assert leaky == [], leaky
print("No target leakage in engineered columns.")